# Trabajo Práctico Diagnóstico

## DDL

![MER](MERBiblioteca.png)

In [ ]:
CREATE DATABASE tpbiblioteca
GO

USE tpbiblioteca

CREATE TABLE Libros(
	ISBN varchar(20) NOT NULL,
	Titulo nvarchar(50) NOT NULL,
	Subtitulo nvarchar(50) NULL,
	Autor nvarchar(50) NULL,
	Editorial nvarchar(50) NOT NULL,
	Categoria nvarchar(50) NULL,
	Subcategoria nvarchar(50) NULL,
	Idioma nvarchar(50) NULL,
	CantPaginas int NULL,
	PlazoMaximo int NULL,
 CONSTRAINT PK_Libros PRIMARY KEY ( ISBN )
)

CREATE TABLE Ejemplares(
	ISBN varchar(20) NOT NULL,
	IDEjemplar char(10) NOT NULL,
	CodigoUbicacion char(10) NULL,
 CONSTRAINT PK_Ejemplares PRIMARY KEY ( ISBN, IDEjemplar )
)
ALTER TABLE Ejemplares  ADD  CONSTRAINT FK_Ejemplares_Libros FOREIGN KEY(ISBN) REFERENCES Libros (ISBN)

CREATE TABLE Usuarios(
	DNI int NOT NULL,
	Apellido nvarchar(50) NULL,
	Nombre nvarchar(50) NULL,
	FechaNacimiento date NULL,
	TipoUsuario nchar(1) NULL,
	Matricula int NULL,
	Promedio numeric(5, 2) NULL,
	AreaEspecialidad nvarchar(50) NULL,
	Oficina nvarchar(50) NULL,
 CONSTRAINT PK_Usuarios PRIMARY KEY ( DNI )
)

CREATE TABLE ReservasPrestamos(
	IDReserva int IDENTITY(1,1) NOT NULL,
	ISBN varchar(20) NOT NULL,
	IDEjemplar char(10) NOT NULL,
	DNI int NOT NULL,
	ReservaFechaDesde date NULL,
	ReservaFechaHasta date NULL,
	PrestamoFechaDesde date NULL,
	PrestamoFechaHasta date NULL,
	DevolucionFecha date NULL,
	Cancelada bit NULL,
 CONSTRAINT PK_ReservasPrestamos PRIMARY KEY ( IDReserva )
)
ALTER TABLE ReservasPrestamos  ADD  CONSTRAINT FK_ReservasPrestamos_Ejemplares FOREIGN KEY(ISBN, IDEjemplar) REFERENCES Ejemplares (ISBN, IDEjemplar)

ALTER TABLE ReservasPrestamos  ADD  CONSTRAINT FK_ReservasPrestamos_Usuarios FOREIGN KEY(DNI) REFERENCES Usuarios (DNI)

CREATE TABLE Amonestaciones(
	IDReserva int NOT NULL,
	AmonestacionFechaDesde date NOT NULL,
	AmonestacionFechaHasta date NOT NULL,
 CONSTRAINT PK_Amonestaciones PRIMARY KEY ( IDReserva )
) 

ALTER TABLE Amonestaciones  ADD CONSTRAINT FK_Amonestaciones_ReservasPrestamos FOREIGN KEY(IDReserva) REFERENCES ReservasPrestamos (IDReserva)


## Datos de Ejemplo

In [ ]:
USE tpbiblioteca

INSERT INTO Libros (ISBN, Titulo, Autor, Editorial, PlazoMaximo) VALUES 
('9781628251845', 'Guía del PMBOK', 'Project Management Institute', 'PMI', 90), -- Requerido consulta D
('0791458601', 'Pensamiento Crítico', 'John Doe', 'Editorial Educativa', 5),     -- Requerido consulta E
('1234567890', 'Martin Fierro', 'Jose Hernandez', 'Plus Ultra', 5),             -- Requerido consulta B
('0987654321', 'Cuentos de la Selva', 'Horacio Quiroga', 'Losada', 5),
('1112223334', 'Geometría Analítica', 'Martin Hernadez', 'Pearson', 90);        -- Para probar LIKE '%Hernadez%'

INSERT INTO Ejemplares (ISBN, IDEjemplar, CodigoUbicacion) VALUES 
('9781628251845', 'EJP-001', 'SEC-A1'),
('0791458601', 'EJP-002', 'SEC-B2'),
('1234567890', 'EJP-003', 'SEC-C1'),
('0987654321', 'EJP-004', 'SEC-D4'),
('1112223334', 'EJP-005', 'SEC-A2');

INSERT INTO Usuarios (DNI, Nombre, Apellido, TipoUsuario, Matricula, AreaEspecialidad) VALUES 
(38987765, 'Juan', 'Perez', 'A', 15420, NULL),          -- Usuario consulta A y E
(12345567, 'Maria', 'Gomez', 'P', NULL, 'Sistemas'),   -- Profesor consulta D
(40123456, 'Pedro', 'Rodriguez', 'A', 16800, NULL),     -- Usuario adicional para listados
(20987654, 'Ana', 'Lopez', 'P', NULL, 'Matemáticas');

-- Préstamo activo para el profesor (Consulta D)
INSERT INTO ReservasPrestamos (ISBN, IDEjemplar, DNI, PrestamoFechaDesde, PrestamoFechaHasta, DevolucionFecha, Cancelada)
VALUES ('9781628251845', 'EJP-001', 12345567, '2026-03-01', '2026-06-01', NULL, 0);

-- Reserva específica sobre ese mismo ejemplar (Consulta D - rango 16/06/2020 al 20/06/2020)
-- Nota: Aunque el año actual sea 2026, la consulta D busca específicamente fechas del 2020.
INSERT INTO ReservasPrestamos (ISBN, IDEjemplar, DNI, ReservaFechaDesde, ReservaFechaHasta, Cancelada)
VALUES ('9781628251845', 'EJP-001', 40123456, '2020-06-18', '2020-06-19', 0);

-- Préstamo activo para el alumno que será una devolución TARDÍA (Consulta E)
-- Se establece una fecha de vencimiento anterior a la fecha actual (28/03/2026) para generar multa.
INSERT INTO ReservasPrestamos (ISBN, IDEjemplar, DNI, PrestamoFechaDesde, PrestamoFechaHasta, DevolucionFecha, Cancelada)
VALUES ('0791458601', 'EJP-002', 38987765, '2026-03-10', '2026-03-15', NULL, 0);

-- Préstamo ya devuelto (para completar historial)
INSERT INTO ReservasPrestamos (ISBN, IDEjemplar, DNI, PrestamoFechaDesde, PrestamoFechaHasta, DevolucionFecha, Cancelada)
VALUES ('1234567890', 'EJP-003', 40123456, '2026-02-01', '2026-02-06', '2026-02-05', 0);

-- Suponiendo que el IDReserva 4 fue una devolución tardía previa
INSERT INTO Amonestaciones (IDReserva, AmonestacionFechaDesde, AmonestacionFechaHasta)
VALUES (4, '2026-02-06', '2026-02-12');

## Consultas

a.	Verificar si la persona con DNI 38.987.765 es un usuario válido, y en caso de serlo devolver como respuesta el tipo de usuario (“Alumno” o “PD”)

In [12]:
SELECT CASE TipoUsuario
WHEN 'A' THEN 'Alumno'
WHEN 'P' THEN 'PDI'
END AS TIPO
FROM Usuarios
WHERE DNI = 38987765;

(Command completed successfully)

b.	Listar el material disponible en la biblioteca, cuyo autor sea “José Hernandez” o cualquier “Hernadez”

In [17]:
SELECT ISBN, Titulo, Subtitulo, Autor, Editorial, 
       Categoria, Subcategoria, Idioma, CantPaginas, PlazoMaximo
FROM Libros
WHERE Autor = 'Jose Hernandez' OR Autor LIKE '%Hernadez%'


(2 rows affected)

ISBN       | Titulo              | Subtitulo | Autor           | Editorial  | Categoria | Subcategoria | Idioma | CantPaginas | PlazoMaximo
-----------+---------------------+-----------+-----------------+------------+-----------+--------------+--------+-------------+------------
1112223334 | Geometría Analítica | NULL      | Martin Hernadez | Pearson    | NULL      | NULL         | NULL   | NULL        | 90         
1234567890 | Martin Fierro       | NULL      | Jose Hernandez  | Plus Ultra | NULL      | NULL         | NULL   | NULL        | 5          
(2 rows)

c.	Listar los usuarios (DNI, Nombre y Apellido) junto con la cantidad de libros que tienen prestados (incluir en el listado TODOS los usuarios).

In [18]:
SELECT U.DNI, U.Nombre, U.Apellido, COUNT(RP.IDReserva) AS LibrosPrestados
FROM Usuarios AS U LEFT JOIN ReservasPrestamos AS RP ON U.DNI = RP.DNI AND RP.DevolucionFecha IS NULL AND RP.PrestamoFechaDesde IS NOT NULL AND RP.Cancelada = 0
GROUP BY U.DNI, U.Nombre, U.Apellido

(4 rows affected)

DNI      | Nombre | Apellido  | LibrosPrestados
---------+--------+-----------+----------------
12345567 | Maria  | Gomez     | 1              
20987654 | Ana    | Lopez     | 0              
38987765 | Juan   | Perez     | 1              
40123456 | Pedro  | Rodriguez | 0              
(4 rows)

d.	Verificar si el libro con ISBN “9781628251845” que tiene en su poder el PD cuyo DNI es 12.345.567 está reservado entre el 16/06/2020 y el 20/06/2020


In [19]:
SELECT IDEjemplar, ReservaFechaDesde, ReservaFechaHasta
FROM ReservasPrestamos
WHERE ISBN = '9781628251845' AND IDEjemplar IN (
	SELECT IDEjemplar
	FROM ReservasPrestamos
	WHERE DNI = 12345567 AND ISBN = '9781628251845' AND DevolucionFecha IS NULL)
AND (ReservaFechaDesde BETWEEN '20200616' AND '20200620') OR 
(ReservaFechaHasta BETWEEN '20200616' AND '20200620')

(1 row affected)

IDEjemplar | ReservaFechaDesde | ReservaFechaHasta
-----------+-------------------+------------------
EJP-001    | 2020-06-18        | 2020-06-19       
(1 row)

e.	En el momento en que el usuario con DNI 38.987.765 devuelve el libro con ISBN “0791458601” verificar si se está realizando una devolución tardía. Para esto en la consulta de debe comparar la fecha actual del sistema, con la fecha en que el libro debería haber sido devuelto. Si la devolución se realiza en tiempo y forma, la consulta deberá devolver 0. Si la devolución fue tardía, la consulta deberá devolver el plazo de castigo a aplicar, es decir, la cantidad de días que se sobrepasó el límite, multiplicado por tres.

In [20]:
SELECT CASE
	WHEN PrestamoFechaHasta < SYSDATETIME() THEN DATEDIFF(day, PrestamoFechaHasta, SYSDATETIME()) * 3
	ELSE 0
	END AS DiasCastigo
FROM ReservasPrestamos
WHERE DNI = 38987765 AND ISBN = '0791458601' AND DevolucionFecha IS NULL

(1 row affected)

DiasCastigo
-----------
42         
(1 row)